In [25]:
import sys
BASE_DIR = "../../.."
sys.path.insert(0, BASE_DIR)

BASE_DIR = "exp"
sys.path.insert(1, BASE_DIR)

import pandas as pd
import numpy as np
import ast
import random
import json
from time import time
import gc
import os
import joblib
import chromadb
from tqdm import tqdm
from dataclasses import dataclass, field
from sentence_transformers import SentenceTransformer
from typing import Dict, List
from dataclasses import dataclass
import pyarrow as pa

random.seed(42)

from src.agents.hosted import CustomAgent
from src.utils import ReaderMetrics

from embedder.ChromaConnector import (ChromaConnection, 
                                      VectorDBConnectionConfig, 
                                      VectorDBInstance)

CONTEXTS_DATASET_PATH = "../../../data/squadv2/contexts.csv"
QA_DATASET_PATH = "../../../data/squadv2/qa_dataset.csv"
AGENT_MODEL_PATH = "../../../models/Qwen/Qwen2.5-7B-Instruct" # "../../../models/Undi95/Meta-Llama-3-8B-Instruct-hf" | "../../../models/Qwen/Qwen2.5-7B-Instruct"

In [ ]:
PARAMS = {
    'version': "3.2",
    'num_samples': 2000,
    'num_contexts': 15,
    'model': AGENT_MODEL_PATH,
    'system_prompt': "You are an AI assistant who helps solve user issues.",
    "item_format": "- [{score}] {document}",
    "user_prompt": 'Answer the question using the available information from the texts in the list below. Each text has a corresponding real-value score of its relevance to the question in square brackets at the beginning. Scores are ranged from 0.0 (the text is not suitable for generating an answer based on it) to 1.0 (the text is suitable for generating an answer based on it). Use this information. Choose texts with high enough relevance scores. If, based on the specified scores, there are no texts in the list that are relevant enough to generate answer based on them, then generate the following answer: "I do not have an answer to your question". Generate answer only in English. Do not duplicate the question in the answer. Generate only the answer to the specified question. Answer need to be short. Do not generate anything extra.',
    "prompt_format": "{user_p}\n\nAvailable information:\n{cnt_list}\n\nQuestion:\n{q}\n\nAnswer:\n",
    'gen_strat': {'max_new_tokens': 1024},
    'stub_answer': "I do not have an answer to your question",
    "scores_dataset_path": "cosine_scores_squad2.csv",
    'vdb_context_info': {'path': "squadv2/chroma/dbs/v3", 'db': {'db': 'squadv2', 'table': 'contexts'}},
    'vdb_query_info': {'path': "squadv2/chroma/dbs/v3", 'db': {'db': 'squadv2', 'table': 'questions'}}
}

METADATA_SAVE_NAME = 'metadata.json'
USER_PROPMTS_SAVE_NAME = 'user_prompts.json'
PARAMS_SAVE_NAME = 'hyperp.json'
GEN_ANSW_SAVE_NAME = 'generation_info.json'
SCORES_SAVE_NAME = 'scores.json'
LOGS_SAVE_DIR = './logs'
META_INFO_DIR_NAME = 'gen_metainfo'

if os.path.exists(f'{LOGS_SAVE_DIR}/v{PARAMS["version"]}'):
    print("Dir exists")
else:
    print("Creating Dir...")
    os.mkdir(f'{LOGS_SAVE_DIR}/v{PARAMS["version"]}')
    os.mkdir(f'{LOGS_SAVE_DIR}/v{PARAMS["version"]}/{META_INFO_DIR_NAME}')

### Подключение к агенту

In [ ]:
agent = CustomAgent(PARAMS['model'], output_logits=False, use_cache=True, output_attentions=False, output_scores=False, output_hidden_states=False)
output = agent.generate(user_prompt="what is wrong with humanity?", system_prompt=PARAMS['system_prompt'], gen_strategy=PARAMS['gen_strat'])
print(output[0])

### Формируем список контекстов для каждого запроса со скорами

In [ ]:
vdb_contexts_conf = VectorDBConnectionConfig(path=PARAMS['vdb_context_info']['path'], db_info=PARAMS['vdb_context_info']['db'])
connector_contexts = ChromaConnection(config=vdb_contexts_conf)
connector_contexts.count_items()

In [ ]:
vdb_query_conf = VectorDBConnectionConfig(path=PARAMS['vdb_query_info']['path'], db_info=PARAMS['vdb_query_info']['db'])
connector_query = ChromaConnection(config=vdb_query_conf)
print(connector_query.count_items())

In [6]:
scores_df = pd.read_csv(PARAMS["scores_dataset_path"])
dataset_df = pd.read_csv(QA_DATASET_PATH)

In [ ]:
scores_df

In [ ]:
dataset_df

In [ ]:
CONTEXTS_LIST_IDS = []
for i in tqdm(range(PARAMS['num_samples'])):
    cur_scores = ast.literal_eval(scores_df['cos_dists'][i])
    cur_contexts = ast.literal_eval(scores_df['contexts_ids'][i])

    cur_list_ids = [(round(1-score, 5), cntx) for score, cntx in zip(cur_scores, cur_contexts)]
    
    CONTEXTS_LIST_IDS.append(cur_list_ids[:PARAMS['num_contexts']])

In [ ]:
CONTEXTS_LIST_IDS[0]

### Готовим промпт

In [ ]:
USER_PROMPTS = []
gc.collect()
for i in tqdm(range(len(CONTEXTS_LIST_IDS))):
    cur_question = connector_query.read([scores_df['query_id'][i]],includes=['documents'])[0].document
    documents_list = []
    for j in range(len(CONTEXTS_LIST_IDS[i])):
        cur_doc = connector_contexts.read([CONTEXTS_LIST_IDS[i][j][1]],includes=['documents'])[0].document
        cur_score = CONTEXTS_LIST_IDS[i][j][0]

        documents_list.append(PARAMS['item_format'].format(score=cur_score, document=cur_doc.strip()))

    documents_list = '\n'.join(documents_list)
    USER_PROMPTS.append(PARAMS['prompt_format'].format(user_p=PARAMS['user_prompt'], cnt_list=documents_list, q=cur_question))

In [12]:
with open(f"{LOGS_SAVE_DIR}/v{PARAMS['version']}/{USER_PROPMTS_SAVE_NAME}", 'w', encoding='utf-8') as fp:
    fp.write(json.dumps(USER_PROMPTS, ensure_ascii=False, indent=1))

# сохраняем конфигурацию эксперимента
with open(f"{LOGS_SAVE_DIR}/v{PARAMS['version']}/{PARAMS_SAVE_NAME}", 'w', encoding='utf-8') as fp:
    fp.write(json.dumps(PARAMS, ensure_ascii=False, indent=1))

In [ ]:
print(USER_PROMPTS[0])

In [ ]:
del scores_df
gc.collect()

### Генерируем ответы на вопросы

In [ ]:
generate_answers = []
display_iter = 100
s_time = time()
for i in tqdm(range(len(USER_PROMPTS))):
    pred_answer, meta_info = agent.generate(user_prompt=USER_PROMPTS[i], system_prompt=PARAMS['system_prompt'], gen_strategy=PARAMS['gen_strat'])
    generate_answers.append(pred_answer)

    # logits = torch.cat(metainfo['logits'], 0).cpu().detach().numpy()
    # logits_int8 = logits.astype('int8') 
    # token_logits = {f"token_{i}": token_logits for i, token_logits in enumerate(logits_int8)}
    # pa_table = pa.table(token_logits)
    # pa.parquet.write_table(pa_table, f"{LOGS_SAVE_DIR}/v{PARAMS['version']}/{META_INFO_DIR_NAME}/logits_{i}.parquet")
    
    if i % display_iter == 0:
        print(f"\n[{i}]: \nGEN: {pred_answer}\nGOLD: {dataset_df['answer'][i]}")
e_time = time()

In [16]:
# сохраняем используемые контексты + сгнерированные ответы
gen_info = []
for i in range(PARAMS['num_samples']):
    formated_contexts = [(float(item[0]), item[1]) for item in CONTEXTS_LIST_IDS[i]]
    cur_item = {'gen_answer': str(generate_answers[i]), 'used_contexts': formated_contexts}
    gen_info.append(cur_item)

with open(f"{LOGS_SAVE_DIR}/v{PARAMS['version']}/{GEN_ANSW_SAVE_NAME}", 'w', encoding='utf-8') as fp:
    fp.write(json.dumps(gen_info, ensure_ascii=False, indent=1))

with open(f"{LOGS_SAVE_DIR}/v{PARAMS['version']}/{METADATA_SAVE_NAME}", 'w', encoding='utf-8') as fp:
    fp.write(json.dumps({'elapsed_time': e_time - s_time}, ensure_ascii=False, indent=1))

### Оцениваем качество

In [17]:
LOADING_VERSION = "3.2"

In [ ]:
import nltk
nltk.download('punkt')
nltk.download('wordnet')

In [19]:
with open(f'{LOGS_SAVE_DIR}/v{LOADING_VERSION}/{GEN_ANSW_SAVE_NAME}','r', encoding='utf8') as fd:
    predicted_answers = list(map(lambda v: v['gen_answer'], json.loads(fd.read())))

In [ ]:
metrics = ReaderMetrics(base_dir="../../..", model_path='en_electra_base')

In [21]:
dataset_df = pd.read_csv(QA_DATASET_PATH)

In [ ]:
target_scores = {
    'BLEU2': [], 'BLEU1': [],
    'ExactMatch': [],'METEOR': [],
    'BertScore': [],
    'Levenshtain': [],
    'ROUGEL': []}

stub_scores = {
    'BLEU2': [], 'BLEU1': [],
    'ExactMatch': [],'METEOR': [],
    'BertScore': [],
    'Levenshtain': [],
    'ROUGEL': []}

show_step = 50

process = tqdm(range(PARAMS['num_samples']))
target_answers =  dataset_df['answer'].to_list()[:PARAMS['num_samples']]
tmp_stub_pred_answers = []
for i in process:
    
    predicted_answer = predicted_answers[i]
    target_answer = target_answers[i]

    target_scores['BLEU1'] += metrics.bleu1([predicted_answer], [target_answer])
    target_scores['BLEU2'] += metrics.bleu2([predicted_answer], [target_answer])
    target_scores['ExactMatch'] += metrics.exact_match([predicted_answer], [target_answer])
    target_scores['METEOR'] += metrics.meteor([predicted_answer], [target_answer])
    target_scores['Levenshtain'] += metrics.levenshtain_score([predicted_answer], [target_answer])
    target_scores['ROUGEL'] += metrics.rougel([predicted_answer], [target_answer])

    stub_pred_answer = predicted_answer
    tmp_stub_pred_answers.append(stub_pred_answer)
    
    stub_scores['BLEU1'] += metrics.bleu1([stub_pred_answer], [PARAMS['stub_answer']])
    stub_scores['BLEU2'] += metrics.bleu2([stub_pred_answer], [PARAMS['stub_answer']])
    stub_scores['ExactMatch'] += metrics.exact_match([stub_pred_answer], [PARAMS['stub_answer']])
    stub_scores['METEOR'] += metrics.meteor([stub_pred_answer], [PARAMS['stub_answer']])
    stub_scores['Levenshtain'] += metrics.levenshtain_score([stub_pred_answer], [PARAMS['stub_answer']])
    stub_scores['ROUGEL'] += metrics.rougel([stub_pred_answer], [PARAMS['stub_answer']])
            
    if i % show_step == 0:
        process.set_postfix({m_name: np.mean(score) for m_name, score in stub_scores.items()})

target_scores = {m_name: round(float(np.mean(score)), 5) for m_name, score in target_scores.items()}
target_scores['BertScore'] = metrics.bertscore(predicted_answers, target_answers)

stub_scores = {m_name: round(float(np.mean(score)), 5) for m_name, score in stub_scores.items()}
stub_scores['BertScore'] = metrics.bertscore(tmp_stub_pred_answers, [PARAMS['stub_answer']]*len(tmp_stub_pred_answers))
stub_scores['elapsed_time_sec'] = round(float(process.format_dict["elapsed"]), 3)

In [ ]:
LOADING_VERSION

In [24]:
with open(f"{LOGS_SAVE_DIR}/v{LOADING_VERSION}/{SCORES_SAVE_NAME}", 'w', encoding='utf-8') as fp:
    fp.write(json.dumps({'target_answers': target_scores, 'stub_answers': stub_scores}, ensure_ascii=False, indent=1))

### Смотрим: встречается ли релевантный контекст

In [26]:
dataset_df = pd.read_csv(QA_DATASET_PATH)
scores_df = pd.read_csv(PARAMS["scores_dataset_path"])

In [31]:
rel_cntx_ids = dataset_df['relevant_context_id'][:PARAMS['num_samples']].tolist()

In [35]:
retrieved_cntx_ids = list(map(lambda item: ast.literal_eval(item), scores_df['contexts_ids'][:PARAMS['num_samples']]))
retrieved_cntx_ids = list(map(lambda items: list(map(lambda item: int(item[2:]), items)), retrieved_cntx_ids))

In [40]:
true_cnt_exist = [true_id in retr_ids for true_id ,retr_ids in zip(rel_cntx_ids, retrieved_cntx_ids)]

In [ ]:
sum(true_cnt_exist)